In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **installs & imports**

In [3]:
!pip install langchain_community

In [4]:
!pip install python-docx unstructured

In [5]:
!pip install google-generativeai

  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl (1.3 MB)
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.18
    Uninstalling google-ai-generativelanguage-0.6.18:
      Successfully uninstalled google-ai-generativelanguage-0.6.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 2.1.10 requires google-ai-generativelanguage<0.7.0,>=0.6.18, but you have google-ai-generativelanguage 0.6.15 which is incompatible.


In [6]:
!pip install langchain-google-genai

  Using cached google_ai_generativelanguage-0.6.18-py3-none-any.whl.metadata (9.8 kB)
Using cached google_ai_generativelanguage-0.6.18-py3-none-any.whl (1.4 MB)
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [7]:
!pip install chromadb

In [8]:
from langchain.chains import RetrievalQA
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.llms import OpenAI, HuggingFacePipeline
from langchain_community.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import SentenceTransformersTokenTextSplitter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.llms import OpenAI
from langchain_community.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings
from langchain.chains.query_constructor.base import StructuredQueryOutputParser
from langchain.retrievers.self_query.chroma import ChromaTranslator
import os
import glob
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

### **loading documents**

In [9]:
# Path to your folder
folder_path = "/content/drive/MyDrive/eva- dataset/english"

# Collect all .docx files (you can add .doc if needed)
file_paths = glob.glob(f"{folder_path}/*.docx")

docs = []
for path in file_paths:
    loader = UnstructuredWordDocumentLoader(path)
    docs.extend(loader.load())

print(f"Loaded {len(docs)} documents")
print(docs[7].page_content[:1000])  # preview first 500 chars


Loaded 8 documents
Aspetovent

Company Name:

 Eva  Pharma for Pharmaceuticals & Medical Appliances.

Trade Name:

 Aspetovent 

Generic Name:   

Caffeine citrate  20 mg  Eq. to Caffeine anhydrous   10 mg  /ml 

Composition:

Each 1 ml solution contains :

Active ingredient :  Caffeine citrate  20 mg (Eq. to Caffeine anhydrous   10 mg)

Inactive ingredients:  Citric acid monohydrate , sodium citrate dihydrate ,water for injection . 

Pharmaceutical form:

Solution for I.V injection

Therapeutic indications:

 Indicated for the short term treatment of apnea of prematurity in infants

between 28 and <33 weeks gestational age.

Posology and Method of administration:

 Prior to initiation of Caffeine citrate , baseline serum levels of caffeine should be measured in infants previously treated with theophylline, since preterm infants metabolize theophylline to caffeine. Likewise, baseline serum levels of caffeine should be measured in infants born to mothers who consumed caffeine prior to d

In [10]:
docs[0].metadata

{'source': '/content/drive/MyDrive/eva- dataset/english/Anselacox.docx'}

###**Split into Chunks**

***by SentenceTransformers***

In [11]:
# Use a sentence-aware splitter
splitter = SentenceTransformersTokenTextSplitter(
    chunk_overlap=50,
    model_name="sentence-transformers/all-mpnet-base-v2"
)

chunks = []

for doc in docs:
    file_name = os.path.basename(doc.metadata["source"]).replace(".docx", "")

    splits = splitter.split_documents([doc])

    for s in splits:
        s.metadata["drug_name"] = file_name
        chunks.append(s)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### **Embedding**

In [12]:
# Initialize SentenceTransformers embeddings
model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
)


/tmp/ipython-input-3641534339.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


### **Vectore store**

In [13]:
# To clear:
vectorstore = None   # reset reference
# Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

vectorstore.persist()
print("Embeddings created with SentenceTransformers!")

Embeddings created with SentenceTransformers!


/tmp/ipython-input-1849167801.py:10: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [14]:
vectorstore._collection.count()

708

### **Retrieval**

***LLM***

In [15]:
llm_model = "gemini-1.5-flash"  # fast & free
llm = ChatGoogleGenerativeAI(model=llm_model, temperature=0.0, google_api_key="###########################" )

***Contextual Compression Retriever***

In [16]:
compressor = LLMChainExtractor.from_llm(llm)

In [17]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
     #base_retriever=vectorstore.as_retriever(search_type="mmr")
    base_retriever=vectorstore.as_retriever()
)

### **RAG Pipeline**

***Memory***

In [18]:
memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key='answer'
    )

/tmp/ipython-input-262215390.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


***Prompt***

In [19]:
PROMPT_template = """
Greeting Handling
If the user says hello, hi, or any greeting, respond with:
"Hello! I'm EVA Pharma's medical assistant. I can help answer questions about our medications, including dosage, side effects, storage instructions, and composition. How can I assist you today?"
You are a medical assistant with access to specific drug documents.
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}

ANSWER:
"""
PROMPT = PromptTemplate(
    template=PROMPT_template,
    input_variables=["context", "question"]
)

***Retrieval Chain***

In [20]:
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=compression_retriever,  # Make sure this is your compression retriever
    memory=memory,
    combine_docs_chain_kwargs={"prompt": PROMPT},
    return_source_documents=True,
    verbose=True,  # Keep verbose to see what's happening
    chain_type="stuff"  # Explicitly set chain type
)

### **API**

In [21]:
!pip install pyngrok

In [22]:
!ngrok config add-authtoken "###############################"


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [23]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Dict, Any, List
import logging
import os
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import LlamaCpp
from langchain.prompts import PromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
import uvicorn
import nest_asyncio
from pyngrok import ngrok


In [30]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryRequest(BaseModel):
    query: str

# Initialize your components
def initialize_components():
    # Use the correct embeddings model that matches your Chroma collection
    # For 768 dimensions, use a model like "sentence-transformers/all-mpnet-base-v2"
    model_name = "sentence-transformers/all-mpnet-base-v2"  # This produces 768-dim embeddings

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': False}
    )

    # Load existing vector store
    persist_directory = "./chroma_db"

    # Check if the vectorstore exists
    if not os.path.exists(persist_directory):
        logger.error(f"Vectorstore directory {persist_directory} does not exist!")
        return None

    try:
        vectorstore = Chroma(
            persist_directory=persist_directory,
            embedding_function=embeddings
        )

        # Test the vectorstore
        test_results = vectorstore.similarity_search("test", k=1)
        logger.info(f"Vectorstore loaded successfully. Test returned {len(test_results)} results.")

    except Exception as e:
        logger.error(f"Error loading vectorstore: {str(e)}")
        return None

    # Initialize LLM
    llm_model = "gemini-1.5-flash"
    try:
        llm = ChatGoogleGenerativeAI(
            model=llm_model,
            temperature=0.0,
            google_api_key="########################"
        )
        # Test the LLM
        test_response = llm.invoke("Hello")
        logger.info("LLM initialized successfully")
    except Exception as e:
        logger.error(f"Error initializing LLM: {str(e)}")
        return None

    # Create compression retriever
    try:
        compressor = LLMChainExtractor.from_llm(llm)
        compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor,
            base_retriever=vectorstore.as_retriever(search_kwargs={"k": 3})
        )
    except Exception as e:
        logger.error(f"Error creating retriever: {str(e)}")
        # Fall back to simple retriever
        compression_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # Create memory
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key='answer'
    )

    # Create custom prompt
    PROMPT_template = """
    Greeting Handling
    If the user says hello, hi, or any greeting, respond with:
    "Hello! I'm EVA Pharma's medical assistant. I can help answer questions about our medications, including dosage, side effects, storage instructions, and composition. How can I assist you today?"
    You are a medical assistant with access to specific drug documents.
    Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer.
    {context}
    Question: {question}

    ANSWER:
    """

    PROMPT = PromptTemplate(
        template=PROMPT_template,
        input_variables=["context", "question"]
    )

    # Create QA chain
    try:
        qa_chain = ConversationalRetrievalChain.from_llm(
            llm=llm,
            retriever=compression_retriever,
            memory=memory,
            combine_docs_chain_kwargs={"prompt": PROMPT},
            return_source_documents=True,
            verbose=True,
            chain_type="stuff"
        )
        logger.info("QA chain created successfully")
    except Exception as e:
        logger.error(f"Error creating QA chain: {str(e)}")
        return None

    return qa_chain

# Initialize components
qa_chain = initialize_components()

@app.post("/eva-chat")
async def eva_chat_post(request: QueryRequest):
    try:
        if not qa_chain:
            raise HTTPException(status_code=500, detail="RAG system not initialized")

        query = request.query.strip()
        logger.info(f"Received query: {query}")

        if not query:
            raise HTTPException(status_code=400, detail="No query provided")

        # Run the query through your RAG chain
        result = qa_chain.invoke({"question": query, "chat_history": []})

        # Extract the response based on the chain's return structure
        response_text = result.get("answer", "No answer found")

        # Extract source documents if available
        source_documents = result.get("source_documents", [])
        sources = []

        for doc in source_documents:
            if hasattr(doc, 'metadata') and 'source' in doc.metadata:
                sources.append(doc.metadata['source'])

        return {
            "query": query,
            "response": response_text,
            "sources": sources
        }

    except Exception as e:
        logger.error(f"Error processing query: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Internal server error: {str(e)}")

@app.get("/health")
async def health_check():
    return {"status": "healthy", "rag_initialized": qa_chain is not None}

# Simple test endpoint
@app.post("/test")
async def test_endpoint(request: QueryRequest):
    return {"query": request.query, "response": f"Test response to: {request.query}"}

if __name__ == "__main__":
    # allow running inside notebook
    nest_asyncio.apply()

    # expose public URL
    public_url = ngrok.connect(8000)
    print("🚀 API is live at:", public_url)

    # run API
    uvicorn.run(app, host="0.0.0.0", port=8000)